# CO 2 Notes - Real-Time Communication Protocols and Scheduling

This notebook contains complete CO 2 notes for **Real-Time Networks Communication and Security for Autonomous Systems**. It covers communication triggering models, CAN, FlexRay, TSN, wireless communication technologies, and real-time scheduling techniques. The notes include textbook-style explanation, architectures, block diagrams, flowcharts, formulas, comparison tables, case studies, and executable examples.

The Python cells are educational models. They do not replace certified protocol analyzers, CAN/FlexRay/TSN hardware tools, or formal schedulability tools.

## 1. Communication Triggering Models

A triggering model defines when a communication action occurs. In autonomous and embedded systems, communication may be released by a clock schedule or by an event. This choice strongly affects determinism, responsiveness, network load, and verification effort.

### Time-Triggered Communication

In time-triggered communication, messages are transmitted at predefined time instants. Every important node follows a shared schedule. The schedule states which message is allowed to transmit in each time slot.

Basic architecture:

```
Global Clock / Synchronized Time
          |
          v
Communication Schedule Table
          |
          v
Time Slot 1 -> Message A
Time Slot 2 -> Message B
Time Slot 3 -> Message C
          |
          v
Receiver processes messages at predictable times
```

Advantages:

- High determinism.
- Predictable latency and jitter.
- Easier worst-case timing analysis.
- Useful for safety-critical control loops.

Limitations:

- Less flexible when unexpected events occur.
- Schedule design can be complex.
- Unused time slots waste bandwidth if the scheduled message has no data.
- Adding a new message may require schedule redesign.

Examples: FlexRay static segment, TSN scheduled traffic, time-triggered control networks, periodic sensor-sampling messages.

### Event-Triggered Communication

In event-triggered communication, messages are transmitted when an event occurs. The event may be a sensor threshold crossing, packet arrival, obstacle detection, brake press, diagnostic request, or fault condition.

Basic architecture:

```
External Event / Sensor Condition
          |
          v
Message Released
          |
          v
Queue / Arbitration / Medium Access
          |
          v
Message Transmitted
          |
          v
Receiver reacts to event
```

Advantages:

- Highly responsive to unexpected events.
- Efficient when events are rare.
- Naturally supports alarms, faults, and emergency messages.
- Flexible when message demand changes.

Limitations:

- Burst events can overload queues.
- Delay and jitter can be less predictable.
- Requires arbitration, prioritization, or congestion control.
- Worst-case timing proof can be harder than time-triggered scheduling.

Examples: CAN bus arbitration, emergency fault messages, asynchronous sensor alerts, aperiodic diagnostic messages.

### Comparison

| Feature | Time-Triggered | Event-Triggered |
|---|---|---|
| Transmission time | Fixed schedule | On event occurrence |
| Determinism | High | Depends on arbitration and load |
| Responsiveness | Lower for unexpected events | High |
| Predictability | Strong | Weaker under bursts |
| Bandwidth usage | Can waste reserved slots | Efficient for rare events |
| Typical use | Control loops, safety schedules | Alarms, sporadic events, diagnostics |

### Flowchart for Choosing a Triggering Model

```
Start
  |
  v
Is the message periodic and safety critical?
  | Yes
  v
Prefer time-triggered or scheduled communication
  |
  v
Can unused slots be tolerated?
  | Yes -> Static schedule
  | No  -> Hybrid schedule

If No:
  |
  v
Is the message rare but urgent?
  | Yes -> Event-triggered with priority/arbitration
  | No  -> Background or best-effort communication
```

### Key Formula

For a time-triggered slot:

```
slot_margin = allocated_slot_time - message_transmission_time
```

For an event-triggered message:

```
response_time = queueing_delay + arbitration_delay + transmission_time + processing_delay
```

Time-triggered communication controls jitter by fixing when messages are sent. Event-triggered communication controls responsiveness by allowing immediate release, but it must handle collisions, contention, and bursts.

In [1]:
# Time-triggered vs event-triggered release demonstration
time_triggered_period = 10  # ms
horizon = 50                # ms
time_triggered_releases = list(range(0, horizon + 1, time_triggered_period))

event_occurrences = [3, 7, 18, 19, 41]  # irregular event times in ms

print("Time-triggered releases (ms):", time_triggered_releases)
print("Event-triggered releases (ms):", event_occurrences)

time_intervals = [time_triggered_releases[i] - time_triggered_releases[i - 1] for i in range(1, len(time_triggered_releases))]
event_intervals = [event_occurrences[i] - event_occurrences[i - 1] for i in range(1, len(event_occurrences))]

print("Time-triggered inter-arrival intervals:", time_intervals)
print("Event-triggered inter-arrival intervals:", event_intervals)
print("Observation: fixed intervals give predictable release timing; event intervals vary.")

Time-triggered releases (ms): [0, 10, 20, 30, 40, 50]
Event-triggered releases (ms): [3, 7, 18, 19, 41]
Time-triggered inter-arrival intervals: [10, 10, 10, 10, 10]
Event-triggered inter-arrival intervals: [4, 11, 1, 22]
Observation: fixed intervals give predictable release timing; event intervals vary.


## 2. Controller Area Network - CAN

Controller Area Network, or CAN, is a broadcast serial communication bus widely used in vehicles and embedded systems. CAN is message-oriented: nodes do not send to a fixed receiver address. Instead, they transmit frames with identifiers. Every node can observe the frame and decide whether the message is relevant.

Bosch developed CAN, and CAN FD is standardized in ISO 11898-1:2015 according to Bosch's CAN protocol information page. Texas Instruments' CAN introduction explains core classical CAN arbitration behavior: bus access is event-driven, arbitration is nondestructive and bit-wise, and lower binary message identifiers have higher priority.

### CAN Architecture

```
+---------+     +------------+     +------------+
| ECU 1   |-----|            |-----| ECU 2      |
| Sensor  |     |  CAN Bus   |     | Controller |
+---------+     | CANH/CANL  |     +------------+
                |            |
+---------+-----|            |-----+------------+
| ECU 3   |     +------------+     | ECU 4      |
| Actuator|                        | Gateway    |
+---------+                        +------------+
```

Each ECU contains an application processor, CAN controller, and CAN transceiver. The controller creates and interprets frames. The transceiver converts controller logic to bus signaling.

### CAN Bus Topology

Classical CAN normally uses a multi-drop bus. Nodes connect to the shared CANH/CANL pair. Since all nodes share the same medium, arbitration is required when multiple nodes transmit at the same time.

```
Termination -- ECU -- ECU -- ECU -- ECU -- Termination
             |     |     |     |
            CANH/CANL shared bus
```

### CAN Message Frame

A simplified classical data frame contains:

```
SOF -> Arbitration Field -> Control Field -> Data Field -> CRC -> ACK -> EOF
```

Important fields:

| Field | Purpose |
|---|---|
| SOF | Start of frame |
| Identifier | Message identity and priority |
| RTR | Distinguishes data frame and remote frame behavior |
| IDE | Distinguishes standard and extended frame format |
| DLC | Data length code |
| Data field | Payload bytes |
| CRC | Error detection |
| ACK | Receiver acknowledgement |
| EOF | End of frame |

### Standard and Extended Identifiers

Standard CAN uses an 11-bit identifier. Extended CAN uses a 29-bit identifier. The identifier is not just a label; it determines arbitration priority. Lower numeric identifiers have higher priority.

```
Standard CAN ID range: 0x000 to 0x7FF
Extended CAN ID range: 0x00000000 to 0x1FFFFFFF
```

### Dominant and Recessive Bits

CAN arbitration relies on dominant and recessive bus states. A dominant bit overwrites a recessive bit on the bus. In common descriptions, logical 0 is dominant and logical 1 is recessive. A node monitors the bus while transmitting. If it transmits recessive but reads dominant, it has lost arbitration and stops transmitting.

### Arbitration Process

Flowchart:

```
Bus idle
  |
  v
Multiple nodes start transmission
  |
  v
Compare identifier bits from MSB to LSB
  |
  v
Did node transmit recessive but read dominant?
  | Yes -> Node loses arbitration and retries later
  | No  -> Node continues
  |
  v
Lowest identifier completes transmission
```

Arbitration is nondestructive because the winning frame is not corrupted. This is valuable for real-time control because the highest-priority message continues immediately.

### CAN Message Flow

```
Application produces signal
  -> CAN controller builds frame
  -> Node waits for idle bus
  -> Arbitration occurs if needed
  -> Winning frame transmitted
  -> Receivers check CRC and ACK
  -> Interested nodes deliver payload to application
```

### CAN Bus Load and Timing

Transmission time:

```
frame_transmission_time = frame_length_bits / bus_bit_rate
```

Bus load:

```
bus_load = sum(frame_time_per_period for all messages) / observation_period
```

Approximate worst-case response intuition:

```
response_time = queueing_delay + blocking_delay + interference_from_higher_priority_frames + transmission_time
```

Exact CAN response-time analysis must account for bit stuffing, frame length, retransmissions, error handling, and higher-priority traffic. In lab answers, state whether the calculation is simplified.

### Hexadecimal CAN Identifiers

CAN identifiers are often written in hexadecimal because it is compact and maps cleanly to binary. Example:

```
0x080 = high priority brake message
0x120 = engine status
0x300 = comfort or body electronics message
```

Since `0x080 < 0x120 < 0x300`, the brake message has highest priority among these examples.

### CAN Case Study - Brake Frame Arbitration

Three ECUs try to transmit simultaneously: brake pressure, engine speed, and window status. The brake frame is assigned the lowest identifier, so it wins arbitration. This design prevents a non-critical comfort message from delaying a safety-critical braking message.

In [2]:
# CAN arbitration and bus-load demonstration
messages = [
    {"name": "brake_pressure", "can_id": 0x080, "bits": 128, "period_ms": 10},
    {"name": "engine_status", "can_id": 0x120, "bits": 128, "period_ms": 20},
    {"name": "window_status", "can_id": 0x300, "bits": 128, "period_ms": 100},
]

winner = min(messages, key=lambda m: m["can_id"])
bit_rate = 500_000  # 500 kbps
observation_ms = 100

print("CAN arbitration winner:", winner["name"], f"ID=0x{winner['can_id']:03X}")

total_bus_time_ms = 0
for msg in messages:
    tx_time_ms = msg["bits"] / bit_rate * 1000
    count = observation_ms / msg["period_ms"]
    total_bus_time_ms += tx_time_ms * count
    print(f"{msg['name']}: ID=0x{msg['can_id']:03X}, tx_time={tx_time_ms:.3f} ms, frames/100ms={count:.1f}")

bus_load = total_bus_time_ms / observation_ms
print(f"Approximate bus load = {bus_load * 100:.2f}%")

CAN arbitration winner: brake_pressure ID=0x080
brake_pressure: ID=0x080, tx_time=0.256 ms, frames/100ms=10.0
engine_status: ID=0x120, tx_time=0.256 ms, frames/100ms=5.0
window_status: ID=0x300, tx_time=0.256 ms, frames/100ms=1.0
Approximate bus load = 4.10%


## 3. FlexRay Protocol

FlexRay is an automotive communication protocol designed for deterministic and fault-tolerant communication. It was created for applications that need higher bandwidth and more predictable timing than classical event-triggered CAN.

FlexRay is based on periodic communication cycles. Vector's FlexRay training material describes a communication cycle as schedule-based and composed of at least a static segment and Network Idle Time, with optional dynamic segment and symbol window. The static segment is used for deterministic transmissions, while the dynamic segment supports event-driven message transmission.

### FlexRay Architecture

```
+-------------+       +----------------------+       +-------------+
| FlexRay ECU | <---> | Channel A / Channel B| <---> | FlexRay ECU |
+-------------+       +----------------------+       +-------------+
       |                        |                          |
       v                        v                          v
Host processor          Communication controller       Bus guardian
```

A FlexRay node typically has a host processor, communication controller, bus driver/transceiver, and possibly a bus guardian. The bus guardian helps prevent faulty nodes from transmitting outside their assigned time.

### Communication Cycle

A FlexRay communication cycle can be represented as:

```
+----------------+-----------------+---------------+--------+
| Static Segment | Dynamic Segment | Symbol Window |  NIT   |
+----------------+-----------------+---------------+--------+
```

Static segment:

- Time-triggered.
- Divided into fixed slots.
- Each slot belongs to a configured message.
- Provides deterministic latency.

Dynamic segment:

- Event-triggered within a controlled cycle region.
- Uses minislot-based access.
- Allows less frequent or event-based messages.

Symbol window:

- Used for protocol symbols such as wake-up or collision-avoidance-related signaling.
- Optional depending on cycle configuration.

Network Idle Time, or NIT:

- No data communication.
- Used for clock correction and synchronization.
- Mandatory with the static segment in basic cycle structure.

### FlexRay Channels

FlexRay supports two channels, usually called channel A and channel B. They may be used for redundancy, increased bandwidth, or fault tolerance. A safety-critical message may be sent on both channels so communication can continue if one channel has a fault.

### FlexRay Timing Formulas

Communication cycle:

```
cycle_time = static_segment_time + dynamic_segment_time + symbol_window_time + NIT_time
```

Static segment time:

```
static_segment_time = number_of_static_slots * static_slot_duration
```

Slot utilization:

```
slot_utilization = payload_transmission_time / allocated_slot_time
```

Effective bandwidth:

```
effective_payload_rate = useful_payload_bits_per_cycle / cycle_time
```

### FlexRay vs CAN

| Feature | CAN | FlexRay |
|---|---|---|
| Triggering | Event-triggered arbitration | Time-triggered static + optional dynamic |
| Medium access | Identifier priority arbitration | Scheduled slots |
| Determinism | Good for high-priority traffic, load-dependent for low-priority | Strong in static segment |
| Fault tolerance | Error detection and retransmission | Dual channels and schedule control |
| Bandwidth | Lower classical CAN bandwidth | Higher than classical CAN |
| Typical use | Body, powertrain, control messages | Safety-critical and deterministic in-vehicle networks |

### FlexRay Case Study - Steer-by-Wire

A steer-by-wire system needs predictable command and feedback timing. The static segment can reserve fixed slots for steering sensor data and actuator commands. This avoids unpredictable arbitration delay and makes timing analysis easier.

In [3]:
# FlexRay cycle timing demonstration
static_slots = 8
static_slot_ms = 0.25
dynamic_segment_ms = 1.0
symbol_window_ms = 0.2
nit_ms = 0.3

static_segment_ms = static_slots * static_slot_ms
cycle_ms = static_segment_ms + dynamic_segment_ms + symbol_window_ms + nit_ms

useful_payload_bits_per_static_slot = 32 * 8
useful_payload_per_cycle = static_slots * useful_payload_bits_per_static_slot
effective_payload_rate_kbps = useful_payload_per_cycle / (cycle_ms / 1000) / 1000

print(f"Static segment = {static_segment_ms:.2f} ms")
print(f"Communication cycle = {cycle_ms:.2f} ms")
print(f"Useful static payload/cycle = {useful_payload_per_cycle} bits")
print(f"Effective static payload rate = {effective_payload_rate_kbps:.2f} kbps")

Static segment = 2.00 ms
Communication cycle = 3.50 ms
Useful static payload/cycle = 2048 bits
Effective static payload rate = 585.14 kbps


## 4. Time-Sensitive Networking - TSN

Time-Sensitive Networking, or TSN, is a family of IEEE 802.1 standards that adds deterministic behavior to Ethernet networks. Standard Ethernet is high-throughput and flexible, but ordinary Ethernet does not automatically guarantee bounded latency or low jitter for safety-critical traffic. TSN adds time synchronization, traffic scheduling, shaping, resource reservation, frame preemption, reliability mechanisms, and management support.

IEEE 802.1AS covers timing and synchronization for time-sensitive applications. IEEE's description states that synchronized time is maintained during normal operation and after network reconfiguration. IEEE 802.1Qbv adds scheduled traffic support through enhancements to bridge forwarding behavior.

### TSN Architecture

```
Talker End Station
     |
     v
TSN Bridge 1 -> TSN Bridge 2 -> TSN Bridge 3
     |              |              |
     v              v              v
Time Sync       Time Sync       Time Sync
     |
     v
Listener End Station
```

Talker: source of a time-sensitive stream.

Listener: receiver of the stream.

Bridge: Ethernet switch with TSN features.

Time synchronization: common clock base across network devices.

### IEEE 802.1AS - Time Synchronization

Time synchronization lets devices agree on a common time base. Scheduled traffic is only meaningful if bridges and end stations agree when the time window begins and ends.

Architecture:

```
Grandmaster Clock
      |
      v
Time Sync Messages
      |
      v
Bridges and End Stations align local clocks
```

### IEEE 802.1Qbv - Time-Aware Shaper

IEEE 802.1Qbv supports scheduled traffic. The Time-Aware Shaper controls output queues using gates. Each gate opens or closes according to a Gate Control List.

Gate Control List concept:

```
Time 0-100 us:   safety queue open, best-effort closed
Time 100-300 us: best-effort open, safety closed
Time 300-400 us: control queue open
Repeat cycle
```

Block diagram:

```
Queue 0: Safety Traffic ---- Gate 0 ----\
Queue 1: Control Traffic --- Gate 1 -----+--> Ethernet Port
Queue 2: Best Effort ------- Gate 2 ----/
```

### Scheduled Traffic and Traffic Shaping

Scheduled traffic reserves transmission windows for critical flows. Traffic shaping controls when frames can leave a port. This reduces interference from non-critical traffic and supports bounded latency.

### TSN Latency Formula

A simplified TSN path delay:

```
path_latency =
    talker_processing
  + sum(bridge_residence_time)
  + sum(link_transmission_time)
  + propagation_delay
  + listener_processing
```

Jitter:

```
jitter = max_latency - min_latency
```

Gate waiting time:

```
gate_wait = next_open_time - arrival_time
```

if a frame arrives before its scheduled window.

### TSN Case Study - Automotive Ethernet Backbone

A vehicle uses Ethernet for camera streams, lidar data, diagnostics, and control messages. Without TSN, large video frames may interfere with critical control messages. With TSN, scheduled windows reserve transmission time for time-critical flows while best-effort traffic uses remaining bandwidth.

In [4]:
# TSN Gate Control List timing demonstration
gcl = [
    {"window": "safety", "start_us": 0, "end_us": 100},
    {"window": "best_effort", "start_us": 100, "end_us": 300},
    {"window": "control", "start_us": 300, "end_us": 400},
]
cycle_us = 400
frame_arrival_us = 260
target_queue = "control"

def next_gate_open(arrival, queue):
    cycle_start = (arrival // cycle_us) * cycle_us
    offset = arrival % cycle_us
    for entry in gcl:
        if entry["window"] == queue and offset <= entry["end_us"]:
            if offset <= entry["start_us"]:
                return cycle_start + entry["start_us"]
            if entry["start_us"] <= offset < entry["end_us"]:
                return arrival
    for entry in gcl:
        if entry["window"] == queue:
            return cycle_start + cycle_us + entry["start_us"]

open_time = next_gate_open(frame_arrival_us, target_queue)
print(f"Frame arrival = {frame_arrival_us} us")
print(f"Target queue = {target_queue}")
print(f"Next gate open = {open_time} us")
print(f"Gate waiting time = {open_time - frame_arrival_us} us")

Frame arrival = 260 us
Target queue = control
Next gate open = 300 us
Gate waiting time = 40 us


## 5. Wireless Communication Technologies

Autonomous systems may use IEEE 802.11/Wi-Fi, LTE, and 5G depending on range, mobility, deployment cost, bandwidth, latency, and reliability requirements.

### IEEE 802.11 / Wi-Fi

IEEE 802.11 is the IEEE working group for wireless LAN standards. IEEE's 802.11 working-group page lists current standards and amendments. Wi-Fi is common, low-cost, and high-throughput over local coverage. However, contention, interference, roaming, and shared spectrum can increase latency and jitter.

Wi-Fi architecture:

```
Station / Vehicle Device
     |
     v
Access Point
     |
     v
Local Network / Edge Server
```

### LTE

LTE is a cellular technology specified through 3GPP. It provides wide-area coverage, mobility support, and managed spectrum operation. LTE is strong for fleet telemetry, monitoring, and cloud connectivity. Tight hard real-time control requires careful measurement because scheduling, handover, radio conditions, and core-network path can vary.

LTE architecture:

```
Vehicle UE -> eNodeB -> EPC/Core Network -> Application Server
```

### 5G

5G is also specified through 3GPP and evaluated globally through IMT-2020 requirements. ITU IMT-2020 requirements include 1 ms user-plane latency for URLLC and 4 ms for eMBB under defined evaluation conditions, plus high reliability requirements for URLLC. These are target/evaluation requirements, not a guarantee that every deployed 5G network provides 1 ms end-to-end latency.

5G architecture:

```
Vehicle UE -> gNodeB -> 5G Core -> MEC / Edge Application
```

MEC, or multi-access edge computing, can reduce delay by placing computation near the radio network.

### Comparison

| Parameter | IEEE 802.11 / Wi-Fi | LTE | 5G |
|---|---|---|---|
| Coverage | Local | Wide area | Wide area, dense small cells possible |
| Mobility support | Moderate, depends on roaming design | Strong | Strong |
| Latency | Can be low but contention-sensitive | Moderate and managed | Lower-latency modes possible |
| Reliability | Affected by shared spectrum | Managed cellular reliability | URLLC features target high reliability |
| Data rate | High locally | Moderate to high | High, depending on spectrum and deployment |
| Best use | Depot, campus, local edge | Fleet telemetry, wide-area monitoring | Low-latency edge, V2X support, high data rate |

### Wireless Timing Formula

```
total_wireless_delay =
    access_delay
  + scheduling_delay
  + transmission_delay
  + retransmission_delay
  + backhaul_delay
  + processing_delay
```

Wireless links are variable, so real-time conclusions should use measured distributions, maximum values, percentiles, and deadline-violation counts.

### Case Study - Remote Driving

Remote driving needs video uplink and control downlink. High data rate alone is not enough. The system must control end-to-end latency, jitter, coverage loss, handover delay, and reliability. A single delayed steering command may be more important than many normal packets.

In [5]:
# Wireless technology comparison using simple weighted scoring
technologies = {
    "Wi-Fi": {"latency": 3, "coverage": 1, "mobility": 2, "data_rate": 3, "predictability": 1},
    "LTE": {"latency": 2, "coverage": 3, "mobility": 3, "data_rate": 2, "predictability": 2},
    "5G": {"latency": 3, "coverage": 2, "mobility": 3, "data_rate": 3, "predictability": 2},
}

weights = {"latency": 0.35, "coverage": 0.20, "mobility": 0.15, "data_rate": 0.15, "predictability": 0.15}

for name, scores in technologies.items():
    weighted = sum(scores[k] * weights[k] for k in weights)
    print(f"{name}: weighted suitability score = {weighted:.2f}/3.00")

print("Note: scores are classroom assumptions, not universal deployment guarantees.")

Wi-Fi: weighted suitability score = 2.15/3.00
LTE: weighted suitability score = 2.35/3.00
5G: weighted suitability score = 2.65/3.00
Note: scores are classroom assumptions, not universal deployment guarantees.


## 6. Real-Time Scheduling Techniques

Scheduling decides which ready task executes next. In real-time systems, the scheduler must support deadline satisfaction, not only fairness or average response time.

### Fixed-Priority Scheduling

In fixed-priority scheduling, each task has a priority that does not change during execution. RMS is the classic fixed-priority policy for periodic tasks.

```
Task Periods -> Static Priority Assignment -> Ready Queue -> Highest Priority Runs
```

### Dynamic-Priority Scheduling

In dynamic-priority scheduling, priorities can change over time. EDF is the classic dynamic-priority policy.

```
Release Time + Deadline -> Current Absolute Deadline -> Ready Queue -> Earliest Deadline Runs
```

### Rate Monotonic Scheduling - RMS

RMS assigns highest priority to the task with the shortest period. A task set is commonly represented as:

```
tau_i = (C_i, T_i, D_i)
```

where `C_i` is computation time, `T_i` is period, and `D_i` is deadline.

CPU utilization:

```
U = sum(C_i / T_i)
```

Liu and Layland RMS sufficient utilization bound:

```
U <= n * (2^(1/n) - 1)
```

This test is sufficient, not necessary. Passing the bound proves schedulability under the model. Failing it means the simple bound is inconclusive; response-time analysis may still show the task set is schedulable.

### Earliest Deadline First - EDF

EDF runs the ready task with the earliest absolute deadline.

Absolute deadline:

```
absolute_deadline = release_time + relative_deadline
```

EDF selection:

```
selected_task = task with minimum absolute_deadline among ready tasks
```

For an ideal preemptive uniprocessor model with independent periodic tasks and deadlines equal to periods, a common utilization condition is:

```
U <= 1
```

### RMS vs EDF

| Feature | RMS | EDF |
|---|---|---|
| Priority type | Fixed | Dynamic |
| Priority basis | Shorter period = higher priority | Earlier absolute deadline = higher priority |
| Simplicity | Simpler | More dynamic bookkeeping |
| Utilization bound | Classic sufficient bound below 1 | Can reach 1 under ideal model |
| Overload behavior | Lower-priority tasks usually miss first | Deadline clustering can cause misses |
| Use case | Periodic embedded control | Mixed deadlines and flexible scheduling |

### Scheduling Flowchart

```
Task released
  |
  v
Add to ready queue
  |
  v
RMS? ------------------ EDF?
  |                      |
  v                      v
Sort by fixed priority   Sort by absolute deadline
  |                      |
  +----------+-----------+
             v
      Run selected task
             |
             v
      Check deadline
```

### Scheduling Case Study - Autonomous Controller

An autonomous controller has a 10 ms steering task, 20 ms braking monitor, 50 ms sensor fusion task, and 100 ms telemetry task. RMS gives highest priority to steering. EDF may temporarily prioritize braking if its absolute deadline is earlier. The correct choice depends on task assumptions, overhead, certification needs, and overload behavior.

In [6]:
# RMS and EDF schedulability checks
import math

tasks = [
    {"name": "steering", "C": 1.0, "T": 10.0, "D": 10.0},
    {"name": "brake_monitor", "C": 1.5, "T": 20.0, "D": 20.0},
    {"name": "sensor_fusion", "C": 3.0, "T": 50.0, "D": 50.0},
    {"name": "telemetry", "C": 2.0, "T": 100.0, "D": 100.0},
]

utilization = sum(t["C"] / t["T"] for t in tasks)
n = len(tasks)
rms_bound = n * (2 ** (1 / n) - 1)

print(f"Total utilization U = {utilization:.3f}")
print(f"RMS sufficient bound for n={n}: {rms_bound:.3f}")
print("RMS bound result:", "PASS" if utilization <= rms_bound else "INCONCLUSIVE")
print("EDF ideal utilization result:", "PASS" if utilization <= 1 else "FAIL")
print("RMS priority order:", [t["name"] for t in sorted(tasks, key=lambda x: x["T"])])

Total utilization U = 0.255
RMS sufficient bound for n=4: 0.757
RMS bound result: PASS
EDF ideal utilization result: PASS
RMS priority order: ['steering', 'brake_monitor', 'sensor_fusion', 'telemetry']


## Consolidated CO 2 Formula Sheet

### Communication Triggering

```
slot_margin = allocated_slot_time - message_transmission_time
event_response_time = queueing_delay + arbitration_delay + transmission_time + processing_delay
```

### CAN

```
higher_priority = lower_numeric_identifier
frame_transmission_time = frame_bits / bus_bit_rate
bus_load = total_transmission_time / observation_time
standard_identifier_bits = 11
extended_identifier_bits = 29
```

### FlexRay

```
cycle_time = static_segment + dynamic_segment + symbol_window + NIT
static_segment = number_of_static_slots * slot_duration
effective_payload_rate = useful_payload_bits_per_cycle / cycle_time
```

### TSN

```
gate_wait = next_open_time - frame_arrival_time
path_latency = processing + bridge_residence + transmission + propagation
jitter = max_latency - min_latency
```

### Wireless

```
total_wireless_delay = access + scheduling + transmission + retransmission + backhaul + processing
reliability = successful_packets / transmitted_packets
packet_loss_rate = lost_packets / transmitted_packets
```

### Scheduling

```
U = sum(C_i / T_i)
RMS_bound = n * (2^(1/n) - 1)
absolute_deadline = release_time + relative_deadline
EDF_condition_under_ideal_model = U <= 1
```

## Architecture and Flowchart Supplement

### 1. Hybrid In-Vehicle Network Architecture

Modern vehicles often combine several networks because no single protocol is ideal for every traffic type.

```
Cameras / Lidar ----> Automotive Ethernet / TSN ----\
                                                    Gateway ECU -> Central Compute
Brake ECU ----------> CAN / CAN FD -----------------/
Steering ECU -------> FlexRay or deterministic link -/
Infotainment -------> Ethernet / Wi-Fi -------------/
Cloud Telemetry ----> LTE / 5G ---------------------/
```

Interpretation: Safety-critical control messages need bounded timing. High-bandwidth sensors need large data capacity. Cloud telemetry needs coverage and mobility. The gateway must prevent low-criticality traffic from delaying safety traffic.

### 2. CAN Arbitration Timing Flow

```
Node A sends ID bits: 0 0 0 1 ...
Node B sends ID bits: 0 0 1 0 ...
Bus value:            0 0 0 ...
                          ^
                          Node B transmitted recessive but read dominant
                          Node B loses arbitration
```

Since dominant 0 overwrites recessive 1, the lower binary identifier continues. This is why a smaller hexadecimal CAN ID has higher priority.

### 3. CAN Latency Components

```
CAN_latency =
    waiting_for_idle_bus
  + arbitration_time
  + blocking_by_lower_priority_frame_already_on_bus
  + interference_from_higher_priority_frames
  + own_frame_transmission_time
  + error_recovery_time_if_any
```

A simplified lab calculation usually includes only transmission time and bus load. A stronger answer states this limitation.

### 4. FlexRay Cycle Layout Diagram

```
Cycle k:
| Slot 1 | Slot 2 | Slot 3 | ... | Dynamic Minislots | Symbol | NIT |
  static deterministic region       controlled flexible region

Cycle k+1 repeats the configured schedule.
```

The static region is useful for control loops because every configured message has a reserved place in the cycle. The dynamic region supports event-based communication without giving up the cycle structure.

### 5. TSN Scheduled Traffic Flow

```
Time Sync Established
      |
      v
Talker sends frame into traffic class
      |
      v
TSN bridge checks gate state
      |
      v
Gate open? ---- No ----> Wait until scheduled window
      |
     Yes
      |
      v
Transmit frame with bounded interference
      |
      v
Listener receives within planned window
```

The key idea is that traffic is not just prioritized; it is scheduled in time.

### 6. Wireless Selection Flowchart

```
Need local high data rate?
  | Yes -> Consider Wi-Fi or local 5G
  | No
  v
Need wide-area mobility?
  | Yes -> Consider LTE or 5G
  | No
  v
Need very low latency and edge processing?
  | Yes -> Consider 5G with edge deployment, then measure
  | No  -> Use telemetry-grade link if deadline allows
```

Do not choose wireless technology by name alone. State the application deadline, coverage area, mobility level, reliability requirement, and deployment assumptions.

### 7. Scheduling Decision Flowchart

```
Are tasks periodic with known periods and WCET?
  | Yes
  v
Is simple fixed-priority design preferred?
  | Yes -> Try RMS and utilization/response-time analysis
  | No
  v
Are deadlines dynamic or mixed?
  | Yes -> Consider EDF under correct assumptions
  | No  -> Use simpler static schedule if possible
```

RMS and EDF are not magic solutions. They require valid execution-time estimates, bounded blocking, and overload handling.

### 8. Protocol Selection Table

| Requirement | Better Candidate | Reason |
|---|---|---|
| Simple low-cost ECU messaging | CAN / CAN FD | Mature arbitration-based vehicle bus |
| Deterministic cyclic control | FlexRay static segment | Reserved time slots and cycle structure |
| High-bandwidth deterministic Ethernet | TSN | Scheduled traffic and time synchronization |
| Local high-speed wireless | Wi-Fi | Local coverage and high data rates |
| Wide-area telemetry | LTE / 5G | Mobility and managed cellular infrastructure |
| Low-latency edge service | 5G with MEC or TSN wired edge | Shorter network path and scheduling support |

## Extended CO 2 Case Studies

### Case Study 1 - CAN Brake Priority

A brake ECU, engine ECU, and comfort ECU request the bus at the same time. The brake frame has ID `0x080`, engine status has ID `0x120`, and window status has ID `0x300`. CAN arbitration selects `0x080` because it is the lowest identifier. The timing lesson is that identifier assignment is a safety design decision.

### Case Study 2 - FlexRay Steer-by-Wire

In steer-by-wire, steering-wheel angle and actuator command messages must be delivered predictably. FlexRay static slots reserve communication time for these messages. The dynamic segment can carry less critical diagnostics. This hybrid design combines deterministic control with event-driven flexibility.

### Case Study 3 - TSN Automotive Ethernet

A vehicle Ethernet network carries camera data, radar summaries, diagnostics, and control messages. Without TSN shaping, large video frames can interfere with critical traffic. With IEEE 802.1AS synchronization and IEEE 802.1Qbv scheduled traffic, bridges can open gates for safety traffic at defined times.

### Case Study 4 - Wireless Remote Monitoring

A fleet of autonomous shuttles sends telemetry through LTE or 5G. LTE may be enough for monitoring. If the application becomes remote driving, the design must verify lower latency, lower jitter, stronger coverage, and failover behavior. Do not assume that the name "5G" alone guarantees hard real-time performance.

### Case Study 5 - RMS vs EDF in a Robot Controller

A robot has periodic motor control, sensor sampling, object detection, and logging tasks. RMS is simple and assigns priority based on period. EDF may accept higher utilization under ideal assumptions, but implementation overhead and overload behavior must be considered. The scheduling policy must match the task model and safety requirement.

## Advanced Analysis Notes for Lab and Exam Writing

### A. How to Analyze a CAN Question

First list all messages, identifiers, frame sizes, bit rate, and periods. Convert hexadecimal identifiers to numeric order. The smallest identifier has the highest priority. Calculate frame transmission time using frame bits divided by bit rate. Then estimate bus load over a common observation window. If bus load is high, mention that lower-priority messages may suffer increased latency.

Example answer structure:

```
Message table -> Priority order -> Transmission time -> Bus load -> Latency interpretation
```

Important limitation: a simplified frame-length calculation ignores bit stuffing, error frames, retransmission, oscillator tolerance, and exact controller behavior. A rigorous answer states this limitation instead of pretending the simplified value is exact.

### B. How to Analyze a FlexRay Question

First draw the communication cycle. Mark the static segment, dynamic segment, symbol window, and Network Idle Time. Then identify which messages belong in static slots and which can use dynamic communication. Safety-critical periodic messages should normally be placed in the static segment because it gives deterministic timing.

Example answer structure:

```
Cycle diagram -> Slot allocation -> Cycle time -> Payload per cycle -> Fault-tolerance explanation
```

If two channels are used, state whether they are used redundantly or independently. Redundant use improves fault tolerance. Independent use can increase bandwidth but does not provide the same level of redundancy.

### C. How to Analyze a TSN Question

First identify talkers, listeners, bridges, traffic classes, and synchronized time. Then draw the queue and gate structure for the Time-Aware Shaper. Explain that the Gate Control List opens and closes queues at configured times. Calculate waiting time if a frame arrives before the correct gate opens.

Example answer structure:

```
Talker/listener path -> Time synchronization -> GCL table -> Gate wait -> Path latency -> Jitter
```

TSN is not a single protocol. It is a family of IEEE 802.1 mechanisms. IEEE 802.1AS provides time synchronization, while IEEE 802.1Qbv provides scheduled traffic support. A precise answer names the function of each standard instead of saying only "TSN gives low latency."

### D. How to Analyze a Wireless Communication Question

Start with the application requirement. A parking-lot robot, highway vehicle, warehouse robot, remote-driving system, and fleet telemetry system do not have the same needs. Compare latency, jitter, data rate, reliability, coverage, mobility support, handover behavior, and infrastructure availability.

Example answer structure:

```
Application deadline -> Coverage requirement -> Mobility requirement -> Technology comparison -> Final selection
```

Wi-Fi may be appropriate for local high-throughput communication. LTE may be appropriate for wide-area telemetry and mobility. 5G may support lower-latency and higher-rate use cases, especially with edge computing, but actual deployment must be measured. Do not claim universal latency values without deployment conditions.

### E. How to Analyze an RMS/EDF Scheduling Question

First write the task table with computation time, period, and deadline. Calculate utilization for each task and total utilization. For RMS, assign higher priority to shorter period tasks and compare total utilization with the Liu-Layland sufficient bound. For EDF, compute absolute deadlines and select the ready task with the earliest absolute deadline.

Example answer structure:

```
Task table -> Utilization -> RMS priority order -> RMS bound -> EDF deadline order -> Conclusion
```

State assumptions. The common RMS and EDF formulas assume idealized models such as independent tasks and known computation times. Shared resources, blocking, non-preemptive sections, interrupts, cache effects, and operating-system overhead can change the result.

### F. Common Mistakes

- Do not confuse bandwidth with throughput.
- Do not confuse latency with jitter.
- Do not say CAN priority belongs to the ECU; it belongs to the message identifier.
- Do not say FlexRay is only time-triggered; it can include static and dynamic parts.
- Do not say TSN is ordinary Ethernet; TSN adds deterministic Ethernet mechanisms.
- Do not say 5G always guarantees 1 ms end-to-end latency.
- Do not say RMS failure of the sufficient bound proves the task set is impossible.
- Do not apply EDF utilization tests without checking the task model.

### G. One-Page Revision Map

```
CO 2
|
+-- Triggering Models
|     +-- Time-triggered: deterministic, scheduled
|     +-- Event-triggered: responsive, burst-sensitive
|
+-- CAN
|     +-- Bus topology, frames, IDs, arbitration, bus load
|
+-- FlexRay
|     +-- Communication cycle, static/dynamic segment, NIT, dual channels
|
+-- TSN
|     +-- Deterministic Ethernet, 802.1AS, 802.1Qbv, GCL, bounded latency
|
+-- Wireless
|     +-- Wi-Fi, LTE, 5G, coverage, mobility, reliability, latency
|
+-- Scheduling
      +-- RMS fixed priority
      +-- EDF dynamic priority
```

## Deep Dive A - CAN Protocol Study Notes

### Why CAN Is Used in Vehicles

CAN was designed for short, reliable, priority-based communication among electronic control units. A vehicle contains many ECUs: engine control, braking, steering, battery management, transmission, body electronics, air conditioning, lighting, and diagnostics. These ECUs must exchange small messages quickly and reliably. CAN is useful because it is multi-master, message-oriented, arbitration-based, and includes strong error-detection mechanisms.

CAN is not address-based in the same way as many computer networks. A CAN frame carries an identifier. Receivers decide whether they care about that identifier. This reduces sender-receiver coupling and supports broadcast communication.

### CAN Node Internal Architecture

```
+--------------------------------------------------+
| ECU Application                                  |
|  - control logic                                 |
|  - sensor processing                             |
|  - diagnostics                                   |
+-----------------------+--------------------------+
                        |
                        v
+--------------------------------------------------+
| CAN Controller                                   |
|  - frame creation                                |
|  - arbitration handling                          |
|  - CRC generation/checking                       |
|  - transmit/receive buffers                      |
+-----------------------+--------------------------+
                        |
                        v
+--------------------------------------------------+
| CAN Transceiver                                  |
|  - converts controller signals to bus levels     |
|  - interfaces with CANH and CANL                 |
+-----------------------+--------------------------+
                        |
                        v
+--------------------------------------------------+
| Physical CAN Bus                                 |
+--------------------------------------------------+
```

### CAN Data Frame Study View

A simplified classical CAN data frame can be studied as:

```
SOF
  -> Arbitration field
  -> Control field
  -> Data field
  -> CRC field
  -> ACK field
  -> EOF
```

The arbitration field contains the identifier and arbitration-related bits. The control field includes information such as data length. The data field carries payload. The CRC field supports error detection. The ACK field allows receivers to acknowledge successful reception.

### Standard vs Extended CAN Identifier

Standard CAN uses 11 identifier bits. Extended CAN uses 29 identifier bits. More identifier bits allow a larger identifier space, but they also add overhead. In both cases, identifier value is tied to arbitration priority.

```
11-bit range: 0x000 to 0x7FF
29-bit range: 0x00000000 to 0x1FFFFFFF
```

A lower value means higher priority. Therefore:

```
0x050 has higher priority than 0x080
0x080 has higher priority than 0x120
0x120 has higher priority than 0x300
```

### Worked Arbitration Example

Suppose three frames start at the same time:

| Message | ID | Binary Prefix | Meaning |
|---|---:|---|---|
| Brake pressure | 0x080 | 00010000000 | Safety-critical |
| Engine status | 0x120 | 00100100000 | Powertrain |
| Door status | 0x300 | 01100000000 | Body electronics |

During arbitration, all nodes transmit identifier bits and monitor the bus. If a node transmits recessive but reads dominant, it loses. The brake frame keeps transmitting because its identifier has dominant bits earlier than the others. The losing nodes do not corrupt the winning frame; they retry later.

### Bus Load Interpretation

Bus load tells how much of the observation interval is consumed by message transmission.

```
bus_load = total_frame_transmission_time / observation_time
```

If bus load is low, messages normally wait less. If bus load is high, lower-priority messages can experience larger delays. Very high bus load is dangerous because error handling, retransmissions, and bursts can push timing beyond deadlines.

### CAN Design Rules

Safety-critical messages should receive lower identifiers. Large or frequent non-critical messages should not be assigned high priority. Diagnostic and comfort messages should not interfere with brake, steering, battery, or powertrain messages. Bus load should be calculated for normal and worst-case traffic. If the network is overloaded, engineers can reduce message frequency, reduce payload, split networks, use gateways, or move to CAN FD, FlexRay, or Ethernet/TSN depending on the requirement.

## Deep Dive B - FlexRay Protocol Study Notes

### Why FlexRay Was Introduced

CAN is efficient and robust, but arbitration delay depends on bus load and message priority. For highly deterministic control, a schedule-based protocol can be easier to analyze. FlexRay was designed for deterministic, fault-tolerant, high-speed automotive communication. It supports time-triggered communication through static slots and also supports event-triggered communication through a dynamic segment.

### FlexRay Node Architecture

```
+--------------------------------------------------+
| Host Processor                                   |
|  - application logic                             |
|  - signal processing                             |
|  - schedule configuration                        |
+-----------------------+--------------------------+
                        |
                        v
+--------------------------------------------------+
| FlexRay Communication Controller                 |
|  - cycle counter                                 |
|  - media access control                          |
|  - static and dynamic segment handling           |
|  - clock synchronization support                 |
+-----------------------+--------------------------+
                        |
                        v
+--------------------------------------------------+
| Bus Driver / Channel Interface                   |
|  - Channel A                                     |
|  - Channel B                                     |
+--------------------------------------------------+
```

### FlexRay Communication Cycle

The communication cycle repeats over time:

```
Cycle 0: | Static | Dynamic | Symbol Window | NIT |
Cycle 1: | Static | Dynamic | Symbol Window | NIT |
Cycle 2: | Static | Dynamic | Symbol Window | NIT |
```

Static segment:

- Contains fixed-length static slots.
- Each configured message has a reserved slot.
- Used for deterministic periodic communication.
- Suitable for control loops such as steering feedback and actuator commands.

Dynamic segment:

- Contains minislots.
- Supports event-driven messages.
- More flexible than static slots.
- Suitable for diagnostics, sporadic status, or less frequent messages.

Symbol window:

- Used for special protocol symbols.
- Not a normal data-transfer region.

Network Idle Time:

- Idle region at the end of the cycle.
- Used for clock synchronization and correction.
- Important because FlexRay depends on coordinated timing.

### Slot Allocation Example

```
Static Slot 1: steering_angle_sensor
Static Slot 2: steering_actuator_command
Static Slot 3: brake_pressure_feedback
Static Slot 4: yaw_rate_sensor
Dynamic Segment: diagnostics and occasional fault messages
Symbol Window: protocol symbol use
NIT: synchronization correction
```

### Fault Tolerance

FlexRay can use two channels. If a message is sent on both channels, the system can tolerate some channel faults. If the channels are used independently, bandwidth increases but redundancy decreases. A proper answer must say how the channels are being used.

### FlexRay Timing Equations

```
cycle_time = static_segment + dynamic_segment + symbol_window + NIT
static_segment = static_slots * static_slot_duration
payload_rate = useful_payload_bits_per_cycle / cycle_time
static_message_period = cycle_time * repetition_factor
```

If a message appears once per cycle, its period equals the communication cycle time. If it appears every second cycle, its period is two cycle times.

### FlexRay Case Study

A steer-by-wire system needs sensor feedback and actuator commands at known times. Static slots guarantee communication opportunities. A diagnostic message can wait for the dynamic segment. This separation prevents diagnostic bursts from delaying steering control.

## Deep Dive C - TSN Study Notes

### Why TSN Matters

Ethernet provides high bandwidth and broad ecosystem support, but ordinary Ethernet does not automatically provide deterministic timing. Time-Sensitive Networking adds IEEE 802.1 mechanisms that allow Ethernet to carry time-critical traffic with bounded latency and low jitter under engineered conditions.

### TSN Layered View

```
+--------------------------------------------------+
| Applications: control, perception, audio/video   |
+--------------------------------------------------+
| Stream reservation and configuration             |
+--------------------------------------------------+
| Traffic classes, queues, gates, shaping          |
+--------------------------------------------------+
| Time synchronization                             |
+--------------------------------------------------+
| Ethernet MAC/PHY and physical links              |
+--------------------------------------------------+
```

### IEEE 802.1AS

IEEE 802.1AS provides timing and synchronization. Time-aware transmission requires clocks to agree. If clocks drift, scheduled windows may not align and frames may miss their reserved transmission opportunities.

```
Grandmaster Clock
     |
     v
Time synchronization messages
     |
     v
Bridges and end stations align local time
```

### IEEE 802.1Qbv

IEEE 802.1Qbv defines scheduled traffic using a Time-Aware Shaper. The shaper controls gates on egress queues. A Gate Control List defines which queue is open during each time interval.

```
Safety Queue      -> Gate -> \
Control Queue     -> Gate ->  +-> Output Port
Best Effort Queue -> Gate -> /
```

Example GCL:

| Time Window | Safety Gate | Control Gate | Best Effort Gate |
|---|---|---|---|
| 0-100 us | Open | Closed | Closed |
| 100-250 us | Closed | Open | Closed |
| 250-500 us | Closed | Closed | Open |

### Guard Bands

A guard band is a protected time before a scheduled window. It prevents a lower-priority frame from beginning transmission so late that it overlaps with the critical traffic window.

```
Best-effort traffic | Guard band | Scheduled safety window
```

### TSN Latency Analysis

```
per_hop_delay = gate_wait + frame_transmission_time + bridge_processing + propagation
path_delay = sum(per_hop_delay for each hop)
jitter = max(path_delay_samples) - min(path_delay_samples)
```

Scheduled traffic reduces jitter by limiting when interference can occur. It does not remove the need for correct configuration. A poor GCL can still create missed deadlines.

### TSN Case Study

An automotive Ethernet backbone carries camera traffic, lidar traffic, and control messages. Camera frames are large and can dominate bandwidth. TSN can reserve precise windows for control traffic and place camera traffic in lower-priority or shaped windows. This improves determinism without abandoning Ethernet.

## Deep Dive D - Wireless Communication Study Notes

### Why Wireless Is Hard for Real-Time Systems

Wireless communication is affected by interference, fading, contention, mobility, handover, coverage gaps, retransmissions, and scheduling. These effects make bounded latency harder than in a controlled wired bus. Wireless can still be useful, but the application must match the timing and reliability that the deployment can actually provide.

### Wi-Fi Study Notes

Wi-Fi is useful for local communication, high data rates, and low deployment cost. It is common in campuses, depots, warehouses, labs, and local edge systems. However, Wi-Fi uses shared spectrum and can suffer contention. More users or interference can increase delay and jitter.

```
Robot / Vehicle -> Wi-Fi Access Point -> Local Edge Server
```

Good use: local monitoring, warehouse robots, lab testbeds, parking-area telemetry.

Risk: unpredictable contention if many stations compete for the same channel.

### LTE Study Notes

LTE provides wide-area cellular mobility and managed spectrum. It is useful for fleet telemetry, remote monitoring, and cloud connectivity. LTE latency is generally more controlled than unmanaged local wireless, but end-to-end delay depends on radio scheduling, backhaul, core network, and server location.

```
Vehicle UE -> eNodeB -> Core Network -> Application Server
```

Good use: city-wide monitoring, logistics telemetry, vehicle health upload.

Risk: strict control deadlines may not be guaranteed without measurement and service design.

### 5G Study Notes

5G supports higher data rates, lower-latency modes, network slicing concepts, and edge-computing deployments. It can support advanced autonomous-system use cases, but the actual performance depends on spectrum, cell density, load, handover, device capability, and edge placement.

```
Vehicle UE -> gNodeB -> 5G Core -> MEC Edge Application
```

Good use: edge-assisted perception, low-latency V2X services, high-bandwidth sensor sharing.

Risk: marketing latency values should not be copied into lab conclusions as universal guarantees.

### Wireless Comparison by Use Case

| Use Case | Wi-Fi | LTE | 5G |
|---|---|---|---|
| Lab simulation | Strong | Possible | Possible |
| Warehouse robot | Strong | Less common | Possible |
| City fleet telemetry | Limited coverage | Strong | Strong |
| Remote driving | Risky unless engineered | Possible with constraints | Stronger candidate with edge |
| Cooperative highway V2X | Limited | Possible | Stronger candidate |
| High-rate sensor sharing | Strong locally | Limited by uplink/load | Strong candidate |

### Wireless Timing Checklist

A good wireless answer must include:

- communication range
- mobility and handover requirement
- latency target
- jitter tolerance
- packet loss tolerance
- data-rate demand
- coverage assumption
- edge or cloud processing location
- fallback behavior when the link degrades

## Deep Dive E - Scheduling Study Notes

### Why Scheduling Is Central

Even if communication is deterministic, tasks must still be scheduled on processors. A vehicle controller can miss a deadline because of CPU overload, interrupt storms, blocking on locks, memory delays, or poor priority assignment. Scheduling and communication must be analyzed together.

### Task Model

```
tau_i = (C_i, T_i, D_i)
```

`C_i` is computation time, usually WCET for hard real-time analysis. `T_i` is period. `D_i` is relative deadline. A job is one instance of a task.

### RMS Detailed Notes

RMS is fixed-priority. Shorter period means higher priority.

```
if T_a < T_b, then priority_a > priority_b
```

Utilization:

```
U_i = C_i / T_i
U_total = sum(U_i)
```

Sufficient bound:

```
U_total <= n * (2^(1/n) - 1)
```

The bound approaches about 0.693 for large task counts. This does not mean CPU usage above 69.3 percent is always impossible. It means the simple sufficient test no longer proves schedulability.

### EDF Detailed Notes

EDF is dynamic-priority. The task with the earliest absolute deadline runs.

```
absolute_deadline = release_time + relative_deadline
```

EDF can schedule any feasible independent preemptive uniprocessor task set under the classic ideal model. But practical systems must account for overhead, non-preemptive sections, blocking, and shared resources.

### Scheduling Timeline Example

```
Time: 0    1    2    3    4    5    6
      |----|----|----|----|----|----|
T1:   [ executes ][done]
T2:             [ waits ][executes]
```

Waiting time occurs when a task is ready but another task is running. Response time includes waiting plus execution.

### Scheduling Failure Modes

- WCET underestimated.
- Too many high-frequency tasks added.
- Low-priority task holds a shared resource needed by high-priority task.
- Interrupt load consumes unmodelled CPU time.
- Communication delay is ignored in end-to-end deadline.
- Logging or diagnostics run at unsafe priority.

### Scheduling Design Advice

Use fixed-priority scheduling when tasks are periodic, stable, and certification simplicity matters. Use EDF when deadlines vary and the implementation can handle dynamic priorities. Always reserve CPU margin for interrupts, operating-system overhead, and communication handling.

## Additional Worked Problems and Case Studies

### Worked Problem 1 - CAN Transmission Time

Given:

```
frame_length = 128 bits
bit_rate = 500000 bits/s
```

Calculation:

```
transmission_time = 128 / 500000 = 0.000256 s = 0.256 ms
```

Interpretation: A single frame is short, but many periodic frames can still create high bus load.

### Worked Problem 2 - FlexRay Cycle Time

Given:

```
static slots = 10
slot duration = 0.2 ms
dynamic segment = 1.0 ms
symbol window = 0.1 ms
NIT = 0.4 ms
```

Calculation:

```
static segment = 10 * 0.2 = 2.0 ms
cycle time = 2.0 + 1.0 + 0.1 + 0.4 = 3.5 ms
```

Interpretation: A static message sent once per cycle has a communication period of 3.5 ms.

### Worked Problem 3 - TSN Gate Waiting Time

A control frame arrives at 260 us. Its gate opens at 300 us.

```
gate_wait = 300 - 260 = 40 us
```

Interpretation: The delay is predictable because the gate schedule is known.

### Worked Problem 4 - RMS Utilization

Given tasks:

```
T1: C=1, T=5
T2: C=2, T=10
T3: C=1, T=20
```

Utilization:

```
U = 1/5 + 2/10 + 1/20 = 0.45
```

For three tasks:

```
RMS_bound = 3 * (2^(1/3) - 1) = 0.779
```

Since `0.45 <= 0.779`, the task set passes the RMS sufficient test under the model.

### Case Study - Mixed Protocol Vehicle

A modern vehicle may use CAN for brake and powertrain messages, Ethernet/TSN for high-bandwidth backbone traffic, wireless for cloud telemetry, and a scheduler inside each ECU. The complete timing path may cross all these systems.

```
Sensor ECU -> CAN -> Gateway -> TSN Ethernet -> Central Compute -> CAN -> Actuator ECU
```

A complete analysis must include:

```
sensing time
CAN transmission and arbitration delay
gateway processing time
TSN scheduled transmission delay
central compute execution time
actuator command bus delay
actuator response time
```

This is the main engineering lesson of CO 2: communication protocols and scheduling algorithms must be selected based on timing, reliability, bandwidth, safety, and security requirements together.

## Long-Answer Bank for CO 2

### 1. Explain time-triggered and event-triggered communication with examples.

Time-triggered communication sends messages according to a predefined schedule. Each message has an assigned time slot or release time. This improves determinism because transmission timing is known before execution. It is suitable for periodic safety-critical messages such as steering angle, brake pressure, motor current, or synchronized sensor samples.

Event-triggered communication sends messages when an event occurs. The event may be a fault, obstacle detection, diagnostic request, threshold crossing, or emergency alert. This improves responsiveness because the system does not wait for a fixed schedule before reporting an unexpected condition. The weakness is that bursts of events can overload queues and increase jitter.

A complete answer should state that time-triggered communication favors predictability, while event-triggered communication favors responsiveness. Many real systems use hybrid communication so periodic control messages remain deterministic while alarms and diagnostics remain flexible.

### 2. Explain CAN arbitration using dominant and recessive bits.

CAN arbitration occurs when multiple nodes begin transmitting at the same time. Every transmitting node also monitors the bus. In common CAN descriptions, logical 0 is dominant and logical 1 is recessive. If a node sends recessive but reads dominant, it knows another node has a higher-priority identifier, so it stops transmitting and retries later.

The process is nondestructive because the winning frame continues without corruption. The frame with the lowest numerical identifier wins because its bit pattern has dominant bits earlier in the arbitration field. This is why assigning CAN identifiers is a timing and safety design activity.

### 3. Explain CAN bus load and why it matters.

CAN bus load is the fraction of time the bus is occupied by message transmission. It can be estimated by summing the transmission time of all frames over an observation interval and dividing by the observation interval.

```
bus_load = total_transmission_time / observation_time
```

High bus load increases waiting time, especially for low-priority messages. If bus load is too high, retransmissions or bursts may cause deadline misses. A strong lab answer should recommend reducing message frequency, reducing payload, splitting networks, improving priorities, or moving heavy traffic to another network.

### 4. Explain FlexRay communication cycle.

FlexRay communication is organized into repeating cycles. Each cycle can include a static segment, dynamic segment, symbol window, and Network Idle Time. The static segment contains fixed slots and supports deterministic time-triggered communication. The dynamic segment supports more flexible event-triggered communication using minislots. The symbol window supports protocol symbols. NIT provides idle time used for synchronization correction.

```
cycle_time = static_segment + dynamic_segment + symbol_window + NIT
```

FlexRay is suitable for applications where predictable communication timing is more important than pure flexibility.

### 5. Compare CAN and FlexRay.

CAN is event-triggered and arbitration-based. It is simple, mature, efficient for small messages, and widely used. Its latency depends on priority and bus load. FlexRay uses a communication cycle with deterministic static slots and optional dynamic communication. It supports higher determinism and fault tolerance through dual channels, but it is more complex.

A precise answer should not say FlexRay is always better. CAN is still appropriate for many control and body electronics functions. FlexRay is better when deterministic cyclic timing and fault tolerance justify the complexity.

### 6. Explain TSN and deterministic Ethernet.

TSN is a set of IEEE 802.1 mechanisms that make Ethernet more deterministic. It does not replace Ethernet; it extends Ethernet behavior so time-sensitive traffic can have bounded delay under configured conditions. IEEE 802.1AS supports synchronized time. IEEE 802.1Qbv supports scheduled traffic through a Time-Aware Shaper and Gate Control List.

TSN is important in vehicles because high-bandwidth sensors such as cameras and lidar can use Ethernet, while critical control messages still need predictable delivery. A TSN bridge can reserve windows for safety traffic and prevent best-effort traffic from interfering during those windows.

### 7. Explain Gate Control List.

A Gate Control List is a schedule that controls whether each egress queue gate is open or closed during a time interval. If the safety queue gate is open, safety frames can transmit. If the best-effort gate is closed, best-effort frames must wait.

```
time 0-100 us: safety open
time 100-250 us: control open
time 250-500 us: best effort open
```

The GCL must be coordinated with synchronized clocks. Without time synchronization, devices may disagree about when gates should open.

### 8. Compare Wi-Fi, LTE, and 5G.

Wi-Fi is useful for local high-data-rate communication but can suffer contention and interference. LTE provides wide-area cellular mobility and managed network access, making it useful for fleet telemetry and monitoring. 5G can provide higher data rates, lower latency modes, and edge-computing support, but actual performance depends on deployment.

A strong answer chooses technology based on the application. Warehouse telemetry may use Wi-Fi. City fleet monitoring may use LTE. Edge-assisted autonomous driving may use engineered 5G with local edge infrastructure. Do not claim that one technology is always best.

### 9. Explain RMS.

Rate Monotonic Scheduling is fixed-priority scheduling for periodic tasks. The shorter the period, the higher the priority. Utilization is:

```
U = sum(C_i / T_i)
```

The Liu-Layland sufficient bound is:

```
U <= n * (2^(1/n) - 1)
```

Passing this bound proves schedulability under the model. Failing it does not prove the task set is impossible; it only means the simple sufficient test is inconclusive.

### 10. Explain EDF.

Earliest Deadline First is dynamic-priority scheduling. The ready task with the earliest absolute deadline runs first.

```
absolute_deadline = release_time + relative_deadline
```

Under the classic ideal preemptive uniprocessor model, EDF can schedule task sets up to full utilization if the assumptions are satisfied. In practice, implementation overhead, blocking, shared resources, interrupt handling, and inaccurate WCET estimates must be considered.

### 11. Compare RMS and EDF for autonomous systems.

RMS is simpler and often easier to certify because priorities are fixed. It is suitable for stable periodic control loops. EDF is more flexible because it reacts to changing deadlines and can use CPU capacity efficiently under ideal assumptions. RMS may be preferred in small embedded controllers. EDF may be attractive for mixed workloads, edge nodes, or systems with varying deadlines.

The correct answer depends on the task model, deadline pattern, overhead, safety requirement, and certification context.

### 12. Complete CO 2 Answer Template

For any protocol question:

```
definition -> architecture -> frame/cycle/queue structure -> timing formula -> case study -> limitations
```

For any scheduling question:

```
task table -> utilization -> priority/deadline order -> schedulability test -> timeline -> conclusion
```

For any wireless question:

```
application requirement -> technology comparison -> latency/jitter/reliability -> coverage/mobility -> final selection
```

Final revision checklist: always draw at least one diagram, always write the timing formula, always identify whether the communication is scheduled or event-driven, always mention the consequence of delay, and always state assumptions. A complete CO 2 answer should connect protocol structure to timing behavior. Protocol names alone are not enough; the explanation must show how arbitration, slots, gates, wireless access, or scheduling decisions affect latency, jitter, throughput, reliability, and deadline satisfaction. End every numerical answer with interpretation, not only calculation. Use precise technical vocabulary consistently.

## Exam and Viva Questions

1. Define time-triggered communication and give an autonomous-system example.
2. Define event-triggered communication and explain why burst arrivals are risky.
3. Compare determinism, responsiveness, and predictability in time-triggered and event-triggered systems.
4. Draw CAN bus topology and explain multi-master communication.
5. List major fields of a classical CAN data frame.
6. Explain 11-bit standard and 29-bit extended CAN identifiers.
7. Why does a lower CAN identifier have higher priority?
8. Explain dominant and recessive bits in CAN arbitration.
9. Calculate CAN frame transmission time for a given frame length and bit rate.
10. Define CAN bus load and explain why high bus load increases latency.
11. Draw the FlexRay communication cycle.
12. Compare FlexRay static and dynamic segments.
13. What is Network Idle Time in FlexRay?
14. How do FlexRay channels support fault tolerance?
15. Explain deterministic Ethernet in TSN.
16. What is the role of IEEE 802.1AS?
17. What is the role of IEEE 802.1Qbv?
18. Explain a Gate Control List.
19. Compare Wi-Fi, LTE, and 5G for autonomous systems.
20. Distinguish fixed-priority and dynamic-priority scheduling.
21. Explain RMS priority assignment.
22. Write the RMS utilization bound.
23. Explain EDF using absolute deadlines.
24. Compare RMS and EDF.

## Glossary and Key Terms

Time-triggered communication: communication released according to a predefined time schedule.

Event-triggered communication: communication released when an event occurs, such as sensor threshold crossing or fault detection.

Determinism: repeatable behavior under defined conditions.

Predictability: ability to analyze timing behavior before deployment.

Responsiveness: ability to react quickly to unexpected events.

CAN bus: shared vehicle communication bus where frames are broadcast and priority is based on message identifier.

CAN standard identifier: 11-bit identifier used in standard CAN frames.

CAN extended identifier: 29-bit identifier used in extended CAN frames.

Dominant bit: bus state that overwrites a recessive bit during CAN arbitration.

Recessive bit: bus state that loses when another node transmits a dominant bit.

Arbitration: process of deciding which CAN frame continues when multiple nodes transmit.

Bus load: fraction of time the communication bus is occupied by frame transmission.

FlexRay static segment: deterministic part of the FlexRay cycle with configured slots.

FlexRay dynamic segment: flexible part of the cycle used for event-driven messages.

Symbol window: part of the FlexRay cycle used for protocol symbols.

Network Idle Time: idle part of the FlexRay cycle used for synchronization and clock correction.

TSN: IEEE 802.1 family of standards for deterministic Ethernet behavior.

IEEE 802.1AS: timing and synchronization standard used to provide common time across TSN devices.

IEEE 802.1Qbv: scheduled traffic enhancement using time-aware shaping.

Gate Control List: schedule that opens and closes output queue gates in a TSN bridge.

Scheduled traffic: traffic transmitted only in assigned time windows.

Latency: delay from sender to receiver.

Jitter: variation in latency.

Throughput: actual useful data delivered per unit time.

RMS: fixed-priority real-time scheduling where shorter-period tasks receive higher priority.

EDF: dynamic-priority scheduling where the task with earliest absolute deadline runs first.

Schedulability: ability of a task set to meet all deadlines under stated assumptions.

Final memory rule for exams: draw the architecture first, write formulas second, calculate third, and interpret the result using real-time consequences last.

## References

- Bosch Semiconductors, CAN Protocols: https://www.bosch-semiconductors.com/products/ip-modules/can-protocols/
- Texas Instruments, *Introduction to the Controller Area Network (CAN)*: https://www.ti.com/lit/sloa101
- Vector Certification, FlexRay Communication Cycle: https://certification.vector.com/mod/page/view.php?id=394
- IEEE 802.1AS Timing and Synchronization page: https://ieee802.org/1/pages/802.1as.html
- IEEE Standards Association, IEEE 802.1AS-2025 overview: https://standards.ieee.org/ieee/802.1AS/11968/
- IEEE 802.1 standards list, including 802.1Qbv scheduled traffic: https://standards.ieee.org/ieee/802.1AS/11968/
- IEEE 802.11 Working Group: https://www.ieee802.org/11/
- 3GPP specifications page: https://www.3gpp.org/specifications-technologies
- ITU IMT-2020 minimum technical performance requirements: https://www.itu.int/en/ITU-R/study-groups/rsg5/rwp5d/imt-2020/Documents/%E2%80%8BS01-1_Draft%20Report%20Requirements%20for%20IMT-2020.pdf
- C. L. Liu and J. W. Layland, real-time scheduling theory: https://dl.acm.org/doi/10.1145/321738.321743